# NN_04 — baseline model

`BUILD_SPEC.md` §7, row 04: encoder + embeddings + FiLM + ListNet (§8), γ/β head initialized near zero. Acceptance: **the model overfits a single quarter to near-zero training loss — a capacity check only, not a result.**

**This notebook produces no strategy performance result.** The model is trained and evaluated on the *same* quarter, in-sample by design, to establish that the architecture can represent the target at all. Nothing here says anything about out-of-sample ranking quality; that is `NN_05` onward.

**Input policy (decided 2026-09-08; `nn_common.transform_features` / `nn_common.to_model_input`).** A **fixed, non-fitted** transform: identity for the five `RANK_NORMALIZED` features (already in `[0, 1]`); deterministic signed log compression `signed_log1p(x) = sign(x) · log1p(|x|)` for all 58 other stock-continuous features. No scaler is fit globally, no per-cycle standardization is used, no cross-cycle statistic is computed, and nothing is clipped. The mask from `NN_01` is preserved. **For the tensor library only**, missing entries are then set to `0.0` — after the mask was computed and after the transform — as a *numeric placeholder, not a statistical imputation*: the mask is concatenated to `X` at the encoder input, so the model always knows which values were missing.

**Architecture (`BUILD_SPEC.md` §8).** Input = concat(transformed `X` (63), `mask` (63), sector embedding (8, vocabulary 10), quarter embedding (4)) = 138 → Linear 128 → GELU → Dropout(0.3) → LayerNorm → Linear 64 → GELU = `h`. FiLM: `m (6,)` → Linear 32 → GELU → Linear 128 → split γ, β (64 each), head initialized near zero; `h' = (1 + γ) ⊙ h + β`. Head: 64 → 32 → GELU → 32 → 1. Loss: ListNet over the whole quarter, target `softmax(2**y − 1)`, as specified — see the target section for what that target is numerically.

Model code lives **in this notebook**; `nn_common.py` holds only path / provenance / device / seed / tensor / normalization utilities.

**Data mode: REAL DATA.** Everything below is computed from the cached real-data slice (`rd.slice_bundle()`, fingerprint verified in `NN_00`). No synthetic, simulated, or approximated value appears anywhere; if the raw exports are absent, every computing cell prints the explicit unavailability notice and does nothing.

Cycle **2023-10-01**, the same cycle as `NN_01`/`NN_02`. Kernel: `atlasquant-venv` (= `.venv/bin/python`).

In [1]:
import nn_common as nc

rd = nc.rd
AVAILABLE = nc.provenance_banner("NN_04_baseline_model.ipynb")

neural_ranker / NN_04_baseline_model.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:42  272

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## The fixed input transform

Printed from source so there is no doubt what runs: an elementwise map with no fitted parameter. The family table shows the raw range against the transformed range; `NaN` counts must be unchanged, the five identity columns byte-identical, and nothing clipped (the transformed extreme of every signed-log column must equal `signed_log1p` of the raw extreme).

In [2]:
import inspect

import numpy as np

print(inspect.getsource(nc.signed_log1p))
print(inspect.getsource(nc.transform_features))
print("INPUT_IDENTITY  ", len(nc.INPUT_IDENTITY), nc.INPUT_IDENTITY)
print("INPUT_SIGNED_LOG", len(nc.INPUT_SIGNED_LOG), "features (every other stock-continuous feature)")
assert nc.INPUT_IDENTITY == nc.RANK_NORMALIZED and nc.INPUT_SIGNED_LOG == nc.PASS_THROUGH
assert set(nc.INPUT_IDENTITY) | set(nc.INPUT_SIGNED_LOG) == set(nc.STOCK_CONTINUOUS)

def signed_log1p(x: np.ndarray) -> np.ndarray:
    """``sign(x) * log1p(|x|)`` elementwise; ``NaN`` stays ``NaN``."""
    return np.sign(x) * np.log1p(np.abs(x))

def transform_features(X: np.ndarray) -> np.ndarray:
    """Apply the fixed input transform to an ``(N, 63)`` ``X`` (columns in
    ``STOCK_CONTINUOUS`` order). Elementwise, so it reads no other row and no
    other cycle; ``NaN`` is preserved wherever it was."""
    if X.ndim != 2 or X.shape[1] != len(STOCK_CONTINUOUS):
        raise ValueError(f"expected an (N, {len(STOCK_CONTINUOUS)}) matrix, got {X.shape}")
    out = np.array(X, dtype=np.float64, copy=True)
    idx = [STOCK_CONTINUOUS.index(name) for name in INPUT_SIGNED_LOG]
    out[:, idx] = signed_log1p(out[:, idx])
    if not np.array_equal(np.isnan(out), np.isnan(X)):
        raise ValueError("the input transform must never create or remove a NaN")
    return out

INPUT_IDENTITY   5 ('market_cap', 'volume', 'net_debt', 'adjusted_net_debt', 'analyst_target_price')
INP

In [3]:
if rd.require_data():
    bundle = rd.slice_bundle()
    CYCLE = bundle.cycles[-2]
    assert CYCLE.quarter_start.isoformat() == "2023-10-01"
    T = nc.bundle_cycle_tensors(bundle, transform=nc.normalize_cycle)[CYCLE.quarter_start]
    XT = nc.transform_features(T.X)

    print("selected cycle ", CYCLE.quarter_start, "cutoff", CYCLE.cutoff, "N =", T.n)
    print()
    families = {"identity (RANK_NORMALIZED)": nc.INPUT_IDENTITY, **nc.FEATURE_FAMILIES}
    header = f"{'family':<32}{'feat':>5}{'NaN':>6} | {'raw min':>11}{'raw median':>12}{'raw max':>11} | {'tr. min':>9}{'tr. median':>12}{'tr. max':>9}"
    print(header)
    print("-" * len(header))
    for family, members in families.items():
        idx = [nc.STOCK_CONTINUOUS.index(n) for n in members]
        raw, tr = T.X[:, idx], XT[:, idx]
        print(
            f"{family:<32}{len(members):>5}{int(np.isnan(tr).sum()):>6} | {np.nanmin(raw):>11.4g}{np.nanmedian(raw):>12.4g}{np.nanmax(raw):>11.4g} | "
            f"{np.nanmin(tr):>9.4f}{np.nanmedian(tr):>12.4f}{np.nanmax(tr):>9.4f}"
        )
        assert int(np.isnan(tr).sum()) == int(np.isnan(raw).sum())
    print()
    print(f"transformed X overall: min {np.nanmin(XT):.4f}, max {np.nanmax(XT):.4f}, NaN {int(np.isnan(XT).sum())} (raw NaN {int(np.isnan(T.X).sum())})")

    # identity columns byte-identical; signed-log columns finite where present; nothing clipped
    id_idx = [nc.STOCK_CONTINUOUS.index(n) for n in nc.INPUT_IDENTITY]
    assert T.X[:, id_idx].tobytes() == XT[:, id_idx].tobytes()
    assert np.array_equal(np.isnan(XT), T.mask)
    assert np.isfinite(XT[~T.mask]).all()
    for name in nc.INPUT_SIGNED_LOG:
        j = nc.STOCK_CONTINUOUS.index(name)
        raw, tr = T.X[:, j], XT[:, j]
        ok = ~np.isnan(raw)
        assert np.array_equal(tr[ok], np.sign(raw[ok]) * np.log1p(np.abs(raw[ok])))
        assert np.nanmax(np.abs(tr)) == nc.signed_log1p(np.array([np.nanmax(np.abs(raw))]))[0]  # extreme preserved: no clipping
    print("identity columns byte-identical, NaN pattern == mask, no clipping: True")

selected cycle  2023-10-01 cutoff 2023-09-30 N = 1493

family                           feat   NaN |     raw min  raw median    raw max |   tr. min  tr. median  tr. max
-----------------------------------------------------------------------------------------------------------------
identity (RANK_NORMALIZED)          5   126 |           0      0.4997          1 |    0.0000      0.4997   1.0000
volatility / beta (NEVER_RANK)      8   339 |      -21.39       11.54      296.1 |   -3.1087      2.5292   5.6941
growth rates                       20   319 |       -4130    0.006955       6417 |   -8.3263      0.0069   8.7669
growth acceleration                 4    82 |       -1342   -0.009326      382.1 |   -7.2028     -0.0093   5.9484
margins                             3    19 |      -148.4      0.1263      7.256 |   -5.0064      0.1190   2.1109
margin changes (bps)                4    42 |  -1.481e+06       20.51  7.619e+05 |  -14.2085      3.0683  13.5436
valuation ratios                 

## Torch inputs and the `NaN` placeholder

`to_model_input` applies the transform, then replaces the `NaN` entries with `0.0`, then moves everything to the device. The count of replaced entries must equal the mask count and the original `NaN` count; every model input must be finite; the mask must still equal the original `NaN` locations exactly. `m` enters FiLM **untransformed** (raw units: a policy choice recorded here — the macro block spans 0.06 .. 31.6 in this slice; no fixed map is applied to it).

In [4]:
if rd.require_data():
    import torch

    DEVICE = nc.resolve_device()
    MI = nc.to_model_input(T, DEVICE)
    print("device used           ", DEVICE)
    print(f"{'tensor':<9}{'shape':<14}{'dtype':<15}{'device':<8}{'finite':>7}")
    for name in ("X", "mask", "sector", "quarter", "m", "y"):
        t = getattr(MI, name)
        print(f"{name:<9}{str(tuple(t.shape)):<14}{str(t.dtype):<15}{str(t.device):<8}{str(bool(torch.isfinite(t).all())):>7}")
    print()
    print("NaNs replaced by the 0.0 placeholder for Torch input:", MI.n_nan_replaced)
    print("mask entries                                      :", int(MI.mask.sum().item()))
    print("original NaN entries in X                         :", int(np.isnan(T.X).sum()))
    print("placeholder entries that are exactly 0.0          :", bool((MI.X[MI.mask == 1] == 0).all().item()))
    print("m (raw, untransformed):", {n: round(float(v), 4) for n, v in zip(nc.MACRO, MI.m.tolist())})

    assert all(bool(torch.isfinite(getattr(MI, n)).all()) for n in ("X", "mask", "sector", "quarter", "m", "y"))
    assert MI.n_nan_replaced == int(MI.mask.sum().item()) == int(np.isnan(T.X).sum()) == 1542
    assert np.array_equal(MI.mask.cpu().numpy().astype(bool), np.isnan(T.X)), "mask must still equal the original NaN locations"
    assert bool((MI.X[MI.mask == 1] == 0).all().item())
    assert MI.m.shape == (6,) and MI.X.shape == (T.n, 63) and MI.mask.shape == (T.n, 63)

device used            mps
tensor   shape         dtype          device   finite
X        (1493, 63)    torch.float32  mps:0      True
mask     (1493, 63)    torch.float32  mps:0      True
sector   (1493,)       torch.int64    mps:0      True
quarter  (1493,)       torch.int64    mps:0      True
m        (6,)          torch.float32  mps:0      True
y        (1493,)       torch.int64    mps:0      True

NaNs replaced by the 0.0 placeholder for Torch input: 1542
mask entries                                      : 1542
original NaN entries in X                         : 1542
placeholder entries that are exactly 0.0          : True
m (raw, untransformed): {'fed_funds_rate': 5.33, 'hy_credit_oas': 3.94, 'ust_10y_yield': 4.5711, 'ust_2y_yield': 5.0436, 'vix': 17.52, 'yield_curve_10y_2y': -0.4725}


## The model (`BUILD_SPEC.md` §8), defined here

`use_film=True` is the model under test; the flag exists so `NN_06` can build the FiLM-removed twin from the same definition and nothing else changes. The FiLM output layer is initialized with weights ~ N(0, 1e-4²) and zero bias, so γ and β start near zero and the network starts as the unconditioned model.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

N_FEATURES = len(nc.STOCK_CONTINUOUS)   # 63
N_MACRO = len(nc.MACRO)                 # 6
SECTOR_VOCAB, SECTOR_DIM = len(nc.SECTOR_VOCABULARY), 8   # 10, 8
QUARTER_VOCAB, QUARTER_DIM = 4, 4                          # fiscal quarter 1..4 -> index 0..3
HIDDEN, LATENT = 128, 64


class NeuralRanker(nn.Module):
    def __init__(self, use_film: bool = True, dropout: float = 0.3):
        super().__init__()
        self.use_film = use_film
        self.sector_emb = nn.Embedding(SECTOR_VOCAB, SECTOR_DIM)
        self.quarter_emb = nn.Embedding(QUARTER_VOCAB, QUARTER_DIM)
        self.encoder = nn.Sequential(
            nn.Linear(2 * N_FEATURES + SECTOR_DIM + QUARTER_DIM, HIDDEN),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(HIDDEN),
            nn.Linear(HIDDEN, LATENT),
            nn.GELU(),
        )
        self.film = nn.Sequential(nn.Linear(N_MACRO, 32), nn.GELU(), nn.Linear(32, 2 * LATENT))
        nn.init.normal_(self.film[-1].weight, std=1e-4)   # gamma/beta head near zero
        nn.init.zeros_(self.film[-1].bias)
        self.head = nn.Sequential(nn.Linear(LATENT, 32), nn.GELU(), nn.Linear(32, 1))

    def film_params(self, m):
        gb = self.film(m)
        return gb[:LATENT], gb[LATENT:]

    def forward(self, X, mask, sector, quarter, m):
        z = torch.cat([X, mask, self.sector_emb(sector), self.quarter_emb(quarter - 1)], dim=1)
        h = self.encoder(z)
        if self.use_film:
            gamma, beta = self.film_params(m)
            h = (1.0 + gamma) * h + beta          # same (gamma, beta) broadcast to every name
        return self.head(h).squeeze(-1)           # one score per instrument


def listnet_target(y):
    # BUILD_SPEC section 8, as specified: softmax over the NDCG gain 2**y - 1, computed in the device dtype
    return torch.softmax(2.0 ** y.to(torch.float32) - 1.0, dim=0)


def listnet_loss(scores, y):
    return -(listnet_target(y) * F.log_softmax(scores, dim=0)).sum()


def target_entropy(y):
    p = listnet_target(y)
    p = p[p > 0]
    return float(-(p * torch.log(p)).sum().item())

In [6]:
if rd.require_data():
    nc.seed_everything(nc.SEEDS[0])
    model = NeuralRanker(use_film=True).to(DEVICE)
    counts = {name: sum(p.numel() for p in module.parameters()) for name, module in
              (("sector_emb", model.sector_emb), ("quarter_emb", model.quarter_emb), ("encoder", model.encoder), ("film", model.film), ("head", model.head))}
    for name, n in counts.items():
        print(f"{name:<12}{n:>8} parameters")
    print(f"{'total':<12}{sum(counts.values()):>8}")
    print("encoder input width:", 2 * N_FEATURES + SECTOR_DIM + QUARTER_DIM, "= 63 + 63 + 8 + 4")

    with torch.no_grad():
        gamma, beta = model.film_params(MI.m)
        print(f"\nFiLM at init on this cycle's m: max|gamma| = {gamma.abs().max().item():.2e}, max|beta| = {beta.abs().max().item():.2e}")
        assert gamma.shape == beta.shape == (LATENT,)
        assert gamma.abs().max().item() < 1e-2 and beta.abs().max().item() < 1e-2, "FiLM head must start near zero"

        model.eval()
        scores = model(MI.X, MI.mask, MI.sector, MI.quarter, MI.m)
        loss0 = listnet_loss(scores, MI.y)
    print("forward pass: scores shape", tuple(scores.shape), "== (N,) =", (T.n,), "| finite:", bool(torch.isfinite(scores).all()))
    print(f"initial ListNet loss (eval mode, untrained): {loss0.item():.4f}  | finite: {bool(torch.isfinite(loss0))}")
    assert scores.shape == (T.n,) and bool(torch.isfinite(scores).all()) and bool(torch.isfinite(loss0))
    assert int(MI.quarter.min()) >= 1 and int(MI.quarter.max()) <= 4 and int(MI.sector.max()) < SECTOR_VOCAB

sector_emb        80 parameters
quarter_emb       16 parameters
encoder        26304 parameters
film            4448 parameters
head            2113 parameters
total          32961
encoder input width: 138 = 63 + 63 + 8 + 4

FiLM at init on this cycle's m: max|gamma| = 4.71e-03, max|beta| = 4.15e-03
forward pass: scores shape (1493,) == (N,) = (1493,) | finite: True
initial ListNet loss (eval mode, untrained): 7.3123  | finite: True


## The ListNet target as specified — and what it is numerically

`softmax(2**y − 1)` applies a softmax to a gain that is already exponential in the grade: the logits are 0, 1, 3, 7, …, 255, 511. The numbers below are the target's mass per grade on this real quarter, in the device dtype (float32, MPS has no float64) and in float64. They are reported here because they determine what "overfitting the target" means and what `NN_05` will be optimizing. Two alternative targets are tabulated **for comparison only** — not used anywhere in this notebook — so the choice can be made on numbers.

In [7]:
if rd.require_data():
    y_cpu = MI.y.cpu()
    grades = list(range(10))
    counts_per_grade = [int((y_cpu == g).sum()) for g in grades]
    p_spec32 = listnet_target(y_cpu)
    gain64 = 2.0 ** y_cpu.double() - 1.0
    p_spec64 = torch.softmax(gain64, dim=0)
    p_gain = gain64 / gain64.sum()
    p_softy = torch.softmax(y_cpu.double(), dim=0)

    def entropy(p):
        p = p[p > 0]
        return float(-(p * torch.log(p)).sum())

    print(f"{'grade':<7}{'names':>6}{'spec, float32':>16}{'spec, float64':>16}{'gain/sum (alt)':>17}{'softmax(y) (alt)':>18}")
    for g, n in zip(grades, counts_per_grade):
        row = f"{g:<7}{n:>6}"
        for p in (p_spec32, p_spec64, p_gain, p_softy):
            row += f"{float(p[y_cpu == g].sum()):>16.3g}" if p is not p_softy else f"{float(p[y_cpu == g].sum()):>18.3g}"
        print(row)
    print()
    print(f"target entropy (loss floor)  spec float32: {entropy(p_spec32):.4f}   spec float64: {entropy(p_spec64):.4f}   "
          f"gain/sum: {entropy(p_gain):.4f}   softmax(y): {entropy(p_softy):.4f}   log(grade-9 count) = {np.log(counts_per_grade[9]):.4f}")
    FLOOR = target_entropy(MI.y)
    print(f"\nthe loss cannot go below the target entropy; on the device the floor is {FLOOR:.4f} nats")
    print(f"mass the specified target puts on grade 9 in float32: {float(p_spec32[y_cpu == 9].sum()):.6f}; on grades 0..8: {float(p_spec32[y_cpu < 9].sum()):.3g}")
    assert abs(float(p_spec32.sum()) - 1.0) < 1e-5

grade   names   spec, float32   spec, float64   gain/sum (alt)  softmax(y) (alt)
0         167               0       1.34e-222               0          8.82e-05
1         147               0       3.21e-222        0.000983          0.000211
2         147               0       2.37e-221         0.00295          0.000574
3         148               0        1.3e-219         0.00693           0.00157
4         147               0       3.86e-216          0.0147           0.00424
5         147               0       3.43e-209          0.0305            0.0115
6         148               0       2.73e-195          0.0624            0.0315
7         147               0       1.69e-167           0.125            0.0852
8         147               0       6.57e-112           0.251             0.232
9         148               1               1           0.506             0.634

target entropy (loss floor)  spec float32: 4.9972   spec float64: 4.9972   gain/sum: 6.3458   softmax(y): 6.0333   log

## Capacity check: overfit one quarter

Protocol, fixed before running: full-batch AdamW (lr 1e-3, weight decay 1e-2 per §8), **2,000 steps** — one quarter is one list, so one step is one epoch — dropout active during optimization exactly as the architecture specifies. Because the target is a distribution, the ListNet loss is bounded below by the target's entropy (the floor above), so *near-zero* is measured as the **excess over the floor**, `KL(P_target ‖ P_model) = loss − floor`, evaluated in eval mode (dropout off) on the same quarter. Pre-set pass criterion: final eval-mode KL below 0.01 nats for every seed. Run over all five seeds (§8: never a single-run number); seed 0 is run twice to measure determinism on the device.

In [8]:
if rd.require_data():
    import time

    STEPS = 2000
    LR, WEIGHT_DECAY = 1e-3, 1e-2
    KL_PASS = 0.01
    CHECKPOINTS = (1, 100, 500, 1000, 2000)

    def overfit(seed, log=False):
        nc.seed_everything(seed)
        net = NeuralRanker(use_film=True).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        initial = None
        t0 = time.perf_counter()
        for step in range(1, STEPS + 1):
            net.train()
            opt.zero_grad()
            loss = listnet_loss(net(MI.X, MI.mask, MI.sector, MI.quarter, MI.m), MI.y)
            loss.backward()
            opt.step()
            if initial is None:
                initial = loss.item()
            if log and step in CHECKPOINTS:
                net.eval()
                with torch.no_grad():
                    ev = listnet_loss(net(MI.X, MI.mask, MI.sector, MI.quarter, MI.m), MI.y).item()
                print(f"    step {step:>5}: train-mode loss {loss.item():.4f}   eval-mode loss {ev:.4f}   KL {ev - FLOOR:.5f}")
        net.eval()
        with torch.no_grad():
            final_eval = listnet_loss(net(MI.X, MI.mask, MI.sector, MI.quarter, MI.m), MI.y).item()
            gamma, beta = net.film_params(MI.m)
        return {
            "seed": seed, "initial": initial, "final_train": loss.item(), "final_eval": final_eval,
            "kl": final_eval - FLOOR, "seconds": time.perf_counter() - t0,
            "gamma_max": gamma.abs().max().item(), "beta_max": beta.abs().max().item(),
        }

    print(f"device {DEVICE} | steps = epochs = {STEPS} | AdamW lr {LR} wd {WEIGHT_DECAY} | floor {FLOOR:.4f} | pass: eval KL < {KL_PASS}")
    print(f"seed {nc.SEEDS[0]} trajectory:")
    results = [overfit(nc.SEEDS[0], log=True)]
    for seed in nc.SEEDS[1:]:
        results.append(overfit(seed))
    repeat = overfit(nc.SEEDS[0])

    print()
    print(f"{'seed':<6}{'initial':>9}{'final train':>13}{'final eval':>12}{'KL':>10}{'KL removed':>12}{'max|gamma|':>12}{'max|beta|':>11}{'seconds':>9}")
    for r in results:
        print(f"{r['seed']:<6}{r['initial']:>9.4f}{r['final_train']:>13.4f}{r['final_eval']:>12.4f}{r['kl']:>10.5f}"
              f"{100 * (1 - r['kl'] / (r['initial'] - FLOOR)):>11.2f}%{r['gamma_max']:>12.3f}{r['beta_max']:>11.3f}{r['seconds']:>9.1f}")
    kls = [r["kl"] for r in results]
    print(f"{'mean':<6}{np.mean([r['initial'] for r in results]):>9.4f}{np.mean([r['final_train'] for r in results]):>13.4f}"
          f"{np.mean([r['final_eval'] for r in results]):>12.4f}{np.mean(kls):>10.5f}")
    print(f"\nseed 0 repeated: final eval loss {results[0]['final_eval']:.6f} vs {repeat['final_eval']:.6f} -> bit-identical on {DEVICE}: {results[0]['final_eval'] == repeat['final_eval']}")
    print(f"all {len(results)} seeds below the pre-set KL criterion ({KL_PASS} nats): {all(k < KL_PASS for k in kls)}")
    assert all(np.isfinite(r["final_eval"]) for r in results)
    assert all(k < KL_PASS for k in kls), "capacity check failed: the model did not fit the quarter's target"

device mps | steps = epochs = 2000 | AdamW lr 0.001 wd 0.01 | floor 4.9972 | pass: eval KL < 0.01
seed 0 trajectory:


    step     1: train-mode loss 7.3129   eval-mode loss 7.2980   KL 2.30076


    step   100: train-mode loss 5.6848   eval-mode loss 5.4120   KL 0.41475


    step   500: train-mode loss 5.0317   eval-mode loss 5.0006   KL 0.00336


    step  1000: train-mode loss 5.0108   eval-mode loss 4.9976   KL 0.00035


    step  2000: train-mode loss 5.0070   eval-mode loss 4.9974   KL 0.00014



seed    initial  final train  final eval        KL  KL removed  max|gamma|  max|beta|  seconds
0        7.3129       5.0070      4.9974   0.00014      99.99%      20.781      6.608      5.0
1        7.3096       4.9994      4.9973   0.00007     100.00%      14.742      5.472      4.9
2        7.3123       5.0223      4.9978   0.00062      99.97%      16.347      7.794      4.8
3        7.3233       5.0032      4.9981   0.00086      99.96%      21.689      8.631      4.9
4        7.3120       5.0024      4.9977   0.00046      99.98%      14.040      5.158      4.8
mean     7.3140       5.0069      4.9976   0.00043

seed 0 repeated: final eval loss 4.997355 vs 4.997295 -> bit-identical on mps: False
all 5 seeds below the pre-set KL criterion (0.01 nats): True


### Determinism of training on the device

`NN_00` showed that init and a forward pass are bit-reproducible under a seed on MPS. Whether a 2,000-step *training* run is, matters for the paired designs in `NN_06`–`NN_07` (same seed, FiLM vs FiLM-removed): any run-to-run noise from the device would sit inside the paired difference. Measured here, not assumed: seed 0 repeated on MPS (from the table above) and on CPU.

In [9]:
if rd.require_data():
    CPU = torch.device("cpu")
    MI_CPU = nc.to_model_input(T, CPU)

    def overfit_on(device, inputs, seed):
        nc.seed_everything(seed)
        net = NeuralRanker(use_film=True).to(device)
        opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        t0 = time.perf_counter()
        for _ in range(STEPS):
            net.train()
            opt.zero_grad()
            listnet_loss(net(inputs.X, inputs.mask, inputs.sector, inputs.quarter, inputs.m), inputs.y).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            final = listnet_loss(net(inputs.X, inputs.mask, inputs.sector, inputs.quarter, inputs.m), inputs.y).item()
        return final, time.perf_counter() - t0

    cpu_runs = [overfit_on(CPU, MI_CPU, nc.SEEDS[0]) for _ in range(2)]
    mps_losses = [results[0]["final_eval"], repeat["final_eval"]]
    print(f"{'device':<8}{'final eval loss, run 1':>24}{'run 2':>12}{'bit-identical':>15}{'seconds/run':>13}")
    print(f"{'mps':<8}{mps_losses[0]:>24.6f}{mps_losses[1]:>12.6f}{str(mps_losses[0] == mps_losses[1]):>15}{results[0]['seconds']:>13.1f}")
    print(f"{'cpu':<8}{cpu_runs[0][0]:>24.6f}{cpu_runs[1][0]:>12.6f}{str(cpu_runs[0][0] == cpu_runs[1][0]):>15}{np.mean([r[1] for r in cpu_runs]):>13.1f}")
    print(f"\nmps run-to-run difference in final eval loss: {abs(mps_losses[0] - mps_losses[1]):.6f} nats; cpu: {abs(cpu_runs[0][0] - cpu_runs[1][0]):.6f}")
    print("both devices reach the same fit to within the KL criterion:", all(abs(v - FLOOR) < KL_PASS for v in mps_losses + [r[0] for r in cpu_runs]))

device    final eval loss, run 1       run 2  bit-identical  seconds/run
mps                     4.997355    4.997295          False          5.0
cpu                     4.997255    4.997255           True          7.2

mps run-to-run difference in final eval loss: 0.000060 nats; cpu: 0.000000
both devices reach the same fit to within the KL criterion: True


## Point-in-time check

Stated explicitly, per `CLAUDE.md` and `BUILD_SPEC.md` §6. What this notebook adds on top of `NN_01`–`NN_03`, and what was checked rather than assumed:

1. **The input transform is elementwise.** It reads no other row and no other cycle: transforming any subset of rows on its own gives byte-identical values to the same rows transformed inside the full matrix. No scaler was fit, no per-cycle or cross-cycle statistic exists anywhere in `transform_features` or `to_model_input`.
2. **FiLM sees only the cycle's own `m`**, the single `(6,)` vector the pipeline broadcast at this cycle's cutoff (`NN_01`, `NN_02`).
3. **In-sample by design.** The capacity check trains and evaluates on the same quarter; that is deliberate, and it is why no number above is a performance result. No early-stopping holdout is used here; when `NN_05` introduces one it must come from *inside* the training window, never at or after the target quarter.

In [10]:
if rd.require_data():
    rng = np.random.default_rng(nc.SEEDS[0])
    rows = np.sort(rng.choice(T.n, size=200, replace=False))
    subset_alone = nc.transform_features(T.X[rows])
    subset_in_full = nc.transform_features(T.X)[rows]
    elementwise = subset_alone.tobytes() == subset_in_full.tobytes()
    print("200 random rows transformed alone vs inside the full matrix, byte-identical:", elementwise)
    assert elementwise
    src = inspect.getsource(nc.transform_features) + inspect.getsource(nc.to_model_input) + inspect.getsource(nc.signed_log1p)
    fitted = [w for w in ("mean(", "std(", "median(", "percentile(", "fit(", "quantile(") if w in src]
    print("statistics computed inside the transform code:", fitted or "none")
    assert not fitted
    assert MI.m.shape == (6,) and np.array_equal(MI.m.cpu().numpy(), T.m.astype(np.float32))
    print("FiLM conditioned on the cycle's own m only: True")
    print("point-in-time checks 1-3: passed")

200 random rows transformed alone vs inside the full matrix, byte-identical: True
statistics computed inside the transform code: none
FiLM conditioned on the cycle's own m only: True
point-in-time checks 1-3: passed


## What this established

Every number below was printed by a cell above in this executed run. **This notebook produced no strategy performance result**: the model was trained and evaluated on the same quarter, in-sample by design, as a capacity check only.

**Selected cycle and inputs.** Cycle 2023-10-01 (cutoff 2023-09-30), N = 1,493, device `mps`. Tensors: `X (1493, 63)`, `mask (1493, 63)`, `sector (1493,)`, `quarter (1493,)`, `m (6,)`, `y (1493,)`; encoder input width 138 = 63 + 63 + 8 + 4.

**The fixed input transform, verified.** Identity on the five `RANK_NORMALIZED` columns (byte-identical to the raw columns, range 0 .. 1); `signed_log1p` on the other 58. Raw extremes of −1.48 × 10⁶ .. 7.62 × 10⁵ (margin changes in bps) become −14.21 .. 13.54; growth rates −4,130 .. 6,417 become −8.33 .. 8.77; volatility/beta −21.4 .. 296.1 become −3.11 .. 5.69; price momentum −0.80 .. 5.77 becomes −0.59 .. 1.91. Transformed `X` spans −14.2085 .. 13.5436. `NaN` count unchanged at 1,542, nothing clipped (every column's transformed extreme equals `signed_log1p` of its raw extreme), and the transform is elementwise (200 random rows transformed alone are byte-identical to the same rows inside the full matrix); no statistic of any kind is computed in the transform code. `m` enters FiLM in raw units (0.06 .. 31.6 across the slice) — a recorded choice, no map applied.

**The `NaN` placeholder, verified.** 1,542 entries were replaced by 0.0 for Torch input — equal to the mask count and to the original `NaN` count — and the mask still equals the original `NaN` locations exactly. All six model input tensors are finite. This is a numeric placeholder for the tensor library, not a statistical imputation: the mask is concatenated to `X`, so the model always knows which entries were missing.

**The model.** 32,961 parameters (encoder 26,304; FiLM 4,448; head 2,113; embeddings 96), exactly the §8 shapes. FiLM at initialization on this cycle's `m`: max|γ| = 4.71 × 10⁻³, max|β| = 4.15 × 10⁻³ — near zero, so training starts from the unconditioned model. The forward pass produces one score per instrument, shape (1493,), finite; the initial ListNet loss is 7.3123 and finite.

**The ListNet target as specified is numerically a top-decile target.** `softmax(2**y − 1)` in float32 (the device dtype) puts mass 1.000000 on the 148 grade-9 names and exactly 0 on grades 0–8 (in float64 the grade-8 mass is 6.6 × 10⁻¹¹²); its entropy — the loss floor — is 4.9972 = log(148). Grades 0–8 therefore contribute no gradient at all: the objective as written trains "top decile vs the rest", not a full-list ordering. For comparison only, the normalized-gain target `(2**y − 1)/Σ` spreads mass 0.506 / 0.251 / 0.125 / … / 0.001 across grades 9 → 1 (entropy 6.3458), and `softmax(y)` 0.634 / 0.232 / 0.085 / … (entropy 6.0333). Neither alternative was trained here. **Consequence, acted on in `NN_05`:** the measured walk-forward does not use this literal target.

**Capacity check passed on all five seeds.** Full-batch AdamW (lr 1e-3, weight decay 1e-2), 2,000 steps = 2,000 epochs, dropout active, ~4.8–5.0 s per run on `mps`. Because the loss is bounded below by the target entropy, "near zero" is the excess over the floor, `KL = eval-mode loss − 4.9972`, with the pass criterion KL < 0.01 nats set before running.

| seed | initial loss | final train-mode loss | final eval-mode loss | KL | excess removed |
|---|---|---|---|---|---|
| 0 | 7.3129 | 5.0070 | 4.9974 | 0.00014 | 99.99% |
| 1 | 7.3096 | 4.9994 | 4.9973 | 0.00007 | 100.00% |
| 2 | 7.3123 | 5.0223 | 4.9978 | 0.00062 | 99.97% |
| 3 | 7.3233 | 5.0032 | 4.9981 | 0.00086 | 99.96% |
| 4 | 7.3120 | 5.0024 | 4.9977 | 0.00046 | 99.98% |
| mean | 7.3140 | 5.0069 | 4.9976 | 0.00043 | |

Seed-0 trajectory (eval-mode KL): 2.30076 at step 1, 0.41475 at 100, 0.00336 at 500, 0.00035 at 1,000, 0.00014 at 2,000. Every seed finished below the criterion (worst: seed 3, KL 0.00086). The architecture can represent this quarter's target; that is all this establishes.

**FiLM after the overfit: a caution for `NN_05`–`NN_06`.** With a single macro state, γ and β are unidentifiable from a plain scale/bias and the fit drove them large: max|γ| 14.0 .. 21.7, max|β| 5.2 .. 8.6 across seeds, from < 0.005 at init, despite weight decay 1e-2. With only 4–12 distinct `m` vectors per training window (`NN_03`), the FiLM path can act as a free per-quarter scale/bias that memorizes quarters rather than learning macro conditioning. The paired ablation in `NN_06` is what will distinguish those.

**Training on `mps` is not bit-reproducible; on `cpu` it is.** Seed 0 repeated on `mps` gave final eval losses 4.997355 vs 4.997295 (difference 0.000060 nats); on `cpu`, 4.997255 vs 4.997255, bit-identical, at 7.2 s per run vs 5.0 s on `mps`. Both devices reach the same fit within the KL criterion. Consequence: a re-run of this notebook reproduces the `mps` numbers above only to about 10⁻⁴ nats in KL (this findings cell is refilled from each run's own outputs); and any same-seed paired comparison on `mps` carries device noise inside the paired difference. **Acted on in `NN_05`:** every measured walk-forward fit runs on `cpu`.

**Point-in-time, checked.** The input transform is elementwise and fits nothing; FiLM saw only this cycle's own `m`; the check is deliberately in-sample and therefore not a result; no early-stopping holdout was used, and when `NN_05` introduces one it must come from inside the training window.